# Bulk disable scenarios

This script is designed to disable all scenarios in all projects, unless the scenario is tagged with the Python variable IGNORE_TAG, which defined below.

When this script disables a scenario, it applies the tag "auto_disabled" to the scenario.

Requirements:
    * DSS version 12.0 or later
    * Python v3.6 or later
    

In [0]:
# ==============================================================================
# Configuration & Input
# ==============================================================================
# set this tag
IGNORE_TAG = "do_not_auto_disable_scenario"

In [0]:
import dataiku
import logging


# ==============================================================================
# Logging Setup
# ==============================================================================
logging.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

def disable_tagged_scenarios(target_tag: str):
    """
    Enumerates all scenarios in all projects.
    If a scenario has the target_tag AND is active:
        1. Disables the scenario (auto-trigger off).
        2. Adds the 'auto_disabled' tag.
    """
    try:
        client = dataiku.api_client()
        project_keys = client.list_project_keys()
        logger.info(f"Found {len(project_keys)} projects to scan.")

        for project_key in project_keys:
            process_project(client, project_key, target_tag)
            
    except Exception as e:
        logger.error(f"Fatal error in main execution: {e}")

def process_project(client, project_key: str, target_tag: str):
    try:
        project = client.get_project(project_key)
        scenarios = project.list_scenarios()
        
        for scenario_item in scenarios:
            scenario_id = scenario_item['id']
            process_scenario(project, scenario_id, target_tag)
            
    except Exception as e:
        logger.warning(f"Failed to process project {project_key}: {e}")

def process_scenario(project, scenario_id: str, target_tag: str):
    """
    Applies logic to a single scenario using raw_settings to avoid AttributeError.
    """
    try:
        scenario = project.get_scenario(scenario_id)
        # https://developer.dataiku.com/latest/api-reference/python/scenarios.html#dataikuapi.dss.scenario.DSSScenarioSettings
        settings = scenario.get_settings()
        settings_raw = settings.get_raw()
        
        # Access the raw dictionary for tags to ensure compatibility
        # Initialize tags list if it doesn't exist in the raw settings
        #if 'tags' not in settings_raw:
        #    settings_raw['tags'] = []
            
        current_tags = settings_raw.get('tags',[])
        
        # Check specific requirements:
        # 1. Currently has a tag that matches the ignore_tag variable
        # 2. Scenario's auto-trigger is enabled (active)
        if settings.active and IGNORE_TAG not in current_tags:
            current_tags.append("auto_disabled")
            # Explicitly assign back to ensure reference is kept (good practice for dicts)
            settings_raw['tags'] = current_tags
            settings_raw['active'] = False

            # Idempotent save
            logger.info(f"Disabling scenario: {scenario_id} in project {project.project_key}")
            settings.save()
        #else:
        #    logger.info(f"Skipping scenario: {scenario_id} in project {project.project_key}")
            
    except Exception as e:
        logger.warning(f"Error processing scenario {scenario_id}: {e}")

# ==============================================================================
# Main Execution
# ==============================================================================
if __name__ == "__main__":
    if len(IGNORE_TAG) > 0:
        disable_tagged_scenarios(IGNORE_TAG)
        logger.info(f"Finished.")
    else:
        logger.warning(f"Error - IGNORE_TAG cannot be blank.")